In [1]:
import duckdb

duckdb.sql("""
COPY (SELECT * FROM read_csv_auto('../data/raw/2019-Oct.csv.gz'))
TO '../data/processed/events_2019_10.parquet' (FORMAT PARQUET)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
path = "'../data/processed/events_2019_10.parquet'"

duckdb.sql(f"""
SELECT COUNT(*) AS n_events,
       COUNT(DISTINCT user_id) AS n_users,
       COUNT(DISTINCT user_session) AS n_sessions,
       MIN(event_time), MAX(event_time)
FROM {path}
""").show()

duckdb.sql(f"""
SELECT event_type, COUNT(*) AS n,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM {path}
GROUP BY event_type
ORDER BY n DESC
""").show()

duckdb.sql(f"""
SELECT ROUND(100.0 * AVG(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END), 2) AS pct_missing_category,
       ROUND(100.0 * AVG(CASE WHEN brand IS NULL THEN 1 ELSE 0 END), 2) AS pct_missing_brand,
       ROUND(100.0 * AVG(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END), 4) AS pct_missing_session
FROM {path}
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬─────────┬────────────┬─────────────────────┬─────────────────────┐
│ n_events │ n_users │ n_sessions │   min(event_time)   │   max(event_time)   │
│  int64   │  int64  │   int64    │      timestamp      │      timestamp      │
├──────────┼─────────┼────────────┼─────────────────────┼─────────────────────┤
│ 42448764 │ 3022290 │    9244421 │ 2019-10-01 00:00:00 │ 2019-10-31 23:59:59 │
└──────────┴─────────┴────────────┴─────────────────────┴─────────────────────┘

┌────────────┬──────────┬────────┐
│ event_type │    n     │  pct   │
│  varchar   │  int64   │ double │
├────────────┼──────────┼────────┤
│ view       │ 40779399 │  96.07 │
│ cart       │   926516 │   2.18 │
│ purchase   │   742849 │   1.75 │
└────────────┴──────────┴────────┘

┌──────────────────────┬───────────────────┬─────────────────────┐
│ pct_missing_category │ pct_missing_brand │ pct_missing_session │
│        double        │      double       │       double        │
├──────────────────────┼─────────────

In [3]:
# Session 層級：有沒有 cart、有沒有 purchase 的交叉表
duckdb.sql(f"""
WITH s AS (
    SELECT user_session,
           MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END) AS has_cart,
           MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS has_purchase
    FROM {path}
    GROUP BY user_session
)
SELECT has_cart, has_purchase, COUNT(*) AS n_sessions,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM s
GROUP BY has_cart, has_purchase
ORDER BY has_cart, has_purchase
""").show()

# 完全重複的列有幾筆
duckdb.sql(f"""
SELECT (SELECT COUNT(*) FROM {path})
     - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM {path})) AS n_duplicate_rows
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬──────────────┬────────────┬────────┐
│ has_cart │ has_purchase │ n_sessions │  pct   │
│  int32   │    int32     │   int64    │ double │
├──────────┼──────────────┼────────────┼────────┤
│        0 │            0 │    8333625 │  90.15 │
│        0 │            1 │     337699 │   3.65 │
│        1 │            0 │     281237 │   3.04 │
│        1 │            1 │     291861 │   3.16 │
└──────────┴──────────────┴────────────┴────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┐
│ n_duplicate_rows │
│      int64       │
├──────────────────┤
│            30220 │
└──────────────────┘

